## Exemplo de interação com IA através de audio

In [2]:
!pip install -q langchain langchain-core langchain-community httpx openAI-whisper IPython gTTS

# As biblioteca necessarias

In [3]:
from langchain_community.chat_models import ChatMaritalk
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts.chat import ChatPromptTemplate
import whisper
from base64 import b64decode
from IPython.display import Audio, display, Javascript
from google.colab import output
from threading import Timer
from gtts import gTTS

# Criando o Script Javascript para Capturar o audio no navegador

In [6]:


RECORD = """
const sleep = tire => new Promise(resolve => setTimeout(resolve, tire))
const b2tex = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.readAsDataURL(blob)
  reader.onloadend = () => resolve(reader.result)
})

var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async () => {
    blob = new Blob(chunks)
    text = await b2tex(blob)
    resolve(text)
  }
  recorder.stop()
})

"""

def record(sec=5):
  print(" ouvindo...")
  display(Javascript(RECORD))
  js_result = output.eval_js("record(%s)" % (sec * 1000))
  audio = b64decode(js_result.split(",")[1])
  file_name = "request_audio.m4a"
  with open(file_name, "wb") as f:
    f.write(audio)

  print("Pronto!")
  return f'/content/{file_name}'

# Iniciando o processo de gravação do audio

In [7]:
record_file = record()
display(Audio( record_file,autoplay=True))

 ouvindo...


<IPython.core.display.Javascript object>

Pronto!


# Transcrevendo o Audio

In [8]:

modelo = whisper.load_model("small")

resposta = modelo.transcribe(record_file,language="portuguese")

print(resposta['text'])

100%|███████████████████████████████████████| 461M/461M [00:05<00:00, 94.6MiB/s]
/usr/local/lib/python3.10/dist-packages/whisper/__init__.py:146: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this exper

 Bom dia Lidia, me descreva as funcionalidades que você pode me ajudar.


## Enviado a Transcriçao do audio para IA

# Obtendo o Retorno da IA em Audio

In [9]:


llm = ChatMaritalk(
    model="sabia-2-small",  # Available models: sabia-2-small and sabia-2-medium
    api_key='',  # Insert your API key here
    temperature=0.7,
    max_tokens=500,
)

output_parser = StrOutputParser()

chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            quero  seguindo as regras abaixo:
            Você é um assistenti virtual que seu nome  Alice , esta aqui para me ajuda com rotinas do RH , como fornecendo informações sobre
            Beneficio, Folha de Pagamento, e ponto Eletronico.

            """,
        ),
        ("human", "{lugar}"),
    ]
)

chain = chat_prompt | llm | output_parser

response = chain.invoke({"lugar": f'{resposta}'})
# print(response)
tts = gTTS(response, lang='pt')
tts.save("audio.mp3")
nome_file_mp3 = "audio.mp3"
display(Audio( nome_file_mp3,autoplay=True))



In [ ]:
print(response)

  Boa tarde! A escala 3x1 é uma configuração de horário de trabalho onde o funcionário trabalha 3 dias seguidos e folga 1. Isso significa que, em um mês, o funcionário terá uma combinação de 3 semanas de trabalho com 1 semana de folga, proporcionando um equilíbrio entre dias úteis e dias de descanso.

Essa escala é comum em diversos setores e pode ser ajustada para atender às necessidades específicas da empresa e dos colaboradores. Algumas vantagens dessa escala incluem a possibilidade de rotações mais frequentes e a chance de os funcionários terem mais tempo livre durante a semana.

No entanto, é importante lembrar que a legislação trabalhista pode ter restrições ou requisitos específicos para a implementação dessa escala, portanto, é recomendável consultar as normas vigentes ou um especialista em RH para garantir a conformidade legal.
